In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "ltv.duckdb"
BRONZE_TABLE = "bronze"   

In [2]:
con = duckdb.connect(str(DB_PATH))
con.sql("SHOW TABLES").show()

bronze = con.sql(f"SELECT * FROM {BRONZE_TABLE}").df()
print(f"Bronze rows: {len(bronze):,}   columns: {len(bronze.columns)}")

##Add more summary stuff, stats, what the column names are, 

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ bronze  │
│ silver  │
└─────────┘

Bronze rows: 1,067,371   columns: 8


In [ ]:
def clean_text(value):
    """Strip spaces and lowercase text, blank text becomes Null"""
    if isinstance(value, str):
        value = value.strip().lower()
        return value if value != "" else None
    return value

silver_df = bronze.copy()   #bronze root table isn't changed

#Only text columns are cleaned
text_cols = [c for c in silver_df.columns
             if not pd.api.types.is_numeric_dtype(silver_df[c])
             and not pd.api.types.is_datetime64_any_dtype(silver_df[c])]
silver_df.columns = [col.strip().replace(' ', '_') for col in silver_df.columns]
silver_df['date_only'] = silver_df['InvoiceDate'].dt.date


changes = {}
for col in text_cols:
    before = silver_df[col]
    after = before.map(clean_text)
    changes[col] = int(((before.astype(str) != after.astype(str)) & before.notna()).sum())
    silver_df[col] = after

#Flag any row that has at least one missing value
silver_df["MissingData"] = silver_df.isna().any(axis=1)
silver_df["CustomerIdExists"] = silver_df["Customer_ID"].notna()

print("PREVIEW (nothing saved yet)")
print(f"Text columns checked: {text_cols}")
print(f"Cells that WILL change: {sum(changes.values()):,}")
for col, n in changes.items():
    print(f"   {col}: {n:,}")

print(f"Rows flagged as having a missing value: {silver_df['MissingData'].sum():,} of {len(silver_df):,}")
missing = silver_df.drop(columns="MissingData").isna().sum()
print(f"Rows with a Customer ID: {silver_df['CustomerIdExists'].sum():,}")
print(f"Rows without one: {(~silver_df['CustomerIdExists']).sum():,}")
print("Missing values per column:")
print(missing[missing > 0])

In [ ]:
answer = input("Apply these changes and save to the silver table? (yes/no): ").strip().lower()

if answer == "yes":
    con.execute("CREATE OR REPLACE TABLE silver AS SELECT * FROM silver_df")
    print(f"Done. Saved {len(silver_df):,} rows to the 'silver' table.")
    print(f"Cells changed: {sum(changes.values()):,}")
    print(f"Rows flagged MissingData: {silver_df['MissingData'].sum():,}")
    
else:
    print("No changes saved.")

con.close()

ConnectionException: Connection Error: Connection already closed!

In [ ]:
# Summary of the table: number of rows, each column's name,
# how many non-missing values it has, and its data type
silver_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 10 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   Invoice           1067371 non-null  str           
 1   StockCode         1067371 non-null  str           
 2   Description       1062989 non-null  str           
 3   Quantity          1067371 non-null  int64         
 4   InvoiceDate       1067371 non-null  datetime64[us]
 5   Price             1067371 non-null  float64       
 6   Customer_ID       824364 non-null   float64       
 7   Country           1067371 non-null  str           
 8   MissingData       1067371 non-null  bool          
 9   CustomerIdExists  1067371 non-null  bool          
dtypes: bool(2), datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 67.2 MB


In [ ]:
silver_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,MissingData,CustomerIdExists
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085.0,united kingdom,False,True
1,489434,79323p,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,False,True
2,489434,79323w,white cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,False,True
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085.0,united kingdom,False,True
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085.0,united kingdom,False,True
